In [ ]:
Libraries and functions requiered for evaluation of the model

In [ ]:
import torch
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
from UNet import UNet  # Make sure this is your U-Net architecture
from dataloader import SolarPanelSegmentationDataset  # Your custom dataset class

In [ ]:
Transformation, loading data and batch size definition

In [ ]:

# Define transformation for preprocessing
transform = transforms.Compose([
    transforms.ToTensor()
])

# Define dataset directories for validation
val_image_dir = r'C:\Users\32esla1mpg\Documents\UNet_PV\val_data\images'
val_mask_dir = r'C:\Users\32esla1mpg\Documents\UNet_PV\val_data\masks'

# Create a validation dataset
val_dataset = SolarPanelSegmentationDataset(val_image_dir, val_mask_dir, transform=transform)
val_dataloader = DataLoader(val_dataset, batch_size=8, shuffle=False)

# Initialize U-Net model
in_channels = 3  # RGB images
out_channels = 1  # Binary segmentation (solar panels or background)
model = UNet(in_channels, out_channels)

# Load trained model
model_path = r'C:\Users\32esla1mpg\Documents\UNet_PV\outputs\trained_PV.pth'
model.load_state_dict(torch.load(model_path))
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()

In [ ]:
Evaluation metrics calculation

In [ ]:
# Function to calculate metrics
def calculate_metrics(outputs, targets):
    TP = torch.logical_and(targets == 1, outputs == 1).sum().item()
    TN = torch.logical_and(targets == 0, outputs == 0).sum().item()
    FP = torch.logical_and(targets == 0, outputs == 1).sum().item()
    FN = torch.logical_and(targets == 1, outputs == 0).sum().item()

    precision = TP / (TP + FP) if (TP + FP) > 0 else 0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0
    accuracy = (TP + TN) / (TP + TN + FP + FN)
    f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    # Calculating the probability of random agreement (Pe)
    total = TP + TN + FP + FN
    Pe = ((TP + FP) * (TP + FN) + (FN + TN) * (FP + TN)) / (total ** 2)

    # Calculating Kappa Coefficient
    kappa = (accuracy - Pe) / (1 - Pe) if (1 - Pe) != 0 else 0

    return accuracy, precision, recall, f1_score, kappa, TP, TN, FP, FN
# Running the validation and calculating metrics
total_accuracy, total_precision, total_recall, total_f1_score, total_kappa = 0, 0, 0, 0, 0
total_TP, total_TN, total_FP, total_FN = 0, 0, 0, 0
with torch.no_grad():
    for inputs, masks, _ in val_dataloader:
        inputs = inputs.to(device)
        masks = masks.to(device)
        outputs = model(inputs)
        outputs = (outputs > 0.5).float()

        # Unpacking all returned values
        accuracy, precision, recall, f1_score, kappa, TP, TN, FP, FN = calculate_metrics(outputs, masks)
        total_accuracy += accuracy
        total_precision += precision
        total_recall += recall
        total_f1_score += f1_score
        total_kappa += kappa
        total_TP += TP
        total_TN += TN
        total_FP += FP
        total_FN += FN

# Averaging metrics
num_batches = len(val_dataloader)
print(f'Average Accuracy: {total_accuracy / num_batches:.4f}')
print(f'Average Precision: {total_precision / num_batches:.4f}')
print(f'Average Recall: {total_recall / num_batches:.4f}')
print(f'Average F1 Score: {total_f1_score / num_batches:.4f}')
print(f'Average Kappa Coefficient: {total_kappa / num_batches:.4f}')
print('Confusion Matrix:')
print(f'TP: {total_TP}, FP: {total_FP}, TN: {total_TN}, FN: {total_FN}')